# Voice AI Pipeline — Learned Endpointer Training on REAL Data (AMI Meeting Corpus) + Phase 2.1/2.2/2.3/0.4 GPU Benchmarks

Runs on **Google Colab, T4 GPU runtime**.

**This notebook trains on REAL audio, not synthetic SLURP splices.** Sections 1-5 use the AMI Meeting Corpus (`edinburghcstr/ami`, cc-by-4.0) — real spontaneous speech with real, reconstructed pause timing (see `scripts/prepare_ami_turntaking.py`'s docstring for exactly how, and its verification against a complete meeting's timestamps before being trusted).

**Honest domain-gap statement, stated once here and not hidden:** AMI is corporate meeting talk, not e-commerce customer support. It was chosen after directly verifying — not assuming — that no free, audio-plus-turn-timing, customer-support-domain dataset exists (see `scripts/prepare_dstc2_audio.py` for the closest domain match found, and why it couldn't be used for labeling). What AMI buys is real pause-timing statistics and real spontaneous speech; it does not buy on-topic content. CANDOR and the roleplay recording (original Phase 1.1/1.2) remain the path to genuine e-commerce audio.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Every section follows the same pattern: task cell(s) → validation/gate cell → save a copy to Google Drive → `git add/commit/push` → a hard gate confirming the push actually landed (local `HEAD` == `origin/master`) before the next section is allowed to proceed. Don't skip the validation cells even when they don't look like they do "real" work — several of them are what caught real bugs earlier in this project (see commit history).


## 0. GPU check

In [ ]:
import torch

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

!nvidia-smi

In [ ]:
# --- validation ---
assert torch.cuda.is_available(), (
    "No CUDA device visible. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session."
)
print("[PASS] CUDA GPU available")

## 0b. Mount Google Drive

Colab's local filesystem is ephemeral — anything not committed to git or copied to Drive is lost when the runtime recycles. Checkpoints and per-section outputs get copied to Drive as a second, independent backup alongside the git commits below (belt and suspenders: git history is the real record, Drive is just insurance against a runtime dying mid-section before a commit lands).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/voice_ai_pipeline_checkpoints"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"Drive checkpoint dir ready: {DRIVE_CHECKPOINT_DIR}")

In [ ]:
# --- validation ---
assert os.path.isdir(DRIVE_CHECKPOINT_DIR), f"{DRIVE_CHECKPOINT_DIR} was not created -- Drive mount likely failed"
print(f"[PASS] {DRIVE_CHECKPOINT_DIR} exists and is writable")

## 0c. Clone the repo

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/varshitthhh/voice-ai-pipeline.git"
REPO_DIR = "voice-ai-pipeline"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    print(f"{REPO_DIR} already exists, pulling latest instead of re-cloning")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
# --- validation ---
import os

expected = ["src", "scripts", "docs", "requirements.txt", "README.md"]
missing = [p for p in expected if not os.path.exists(p)]
assert not missing, f"missing expected repo paths: {missing} -- clone/cd did not land where expected"

assert os.path.exists("scripts/prepare_ami_turntaking.py"), (
    "scripts/prepare_ami_turntaking.py not found -- this clone predates the real-data pipeline; "
    "git pull did not bring in the latest commits"
)
print(f"[PASS] repo structure looks right, cwd={os.getcwd()}")

## 0d. Install dependencies

Colab's T4 runtime ships a CUDA-enabled `torch` preinstalled — nothing in `requirements.txt` pins a specific torch version, so this should not downgrade it to a CPU build. If pip ever does touch torch, rerun the GPU check cell above before continuing.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# --- validation ---
import importlib

required = ["torch", "faster_whisper", "silero_vad", "soundfile", "pydantic", "numpy", "pandas", "matplotlib", "scipy"]
missing = []
for mod in required:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)
assert not missing, f"failed to import: {missing}"

import torch
assert torch.cuda.is_available(), "torch lost CUDA after installing requirements.txt -- check for a torch version pin"
print("[PASS] all required packages import cleanly, CUDA still available")

## Push authentication (once for the rest of this notebook)

Colab's git clone has no stored push credentials. Paste a GitHub Personal Access Token with `repo` scope (classic) or `Contents: Read and write` on this repo (fine-grained). Input is hidden via `getpass` so it never appears in cell output.

In [ ]:
from getpass import getpass

GITHUB_TOKEN = getpass("GitHub Personal Access Token (repo scope, input hidden): ")
subprocess.run(
    ["git", "remote", "set-url", "origin", f"https://{GITHUB_TOKEN}@github.com/varshitthhh/voice-ai-pipeline.git"],
    check=True,
)
print("remote URL updated with token for this session's pushes")

In [ ]:
# --- validation: token actually authenticates, before sinking time into training below ---
result = subprocess.run(["git", "fetch", "origin"], capture_output=True, text=True)
assert result.returncode == 0, f"git fetch failed -- token may be invalid, expired, or missing repo scope:\n{result.stderr}"
print("[PASS] token authenticates successfully")

## 1. AMI data pipeline — real turn-taking data

Regenerates the real-audio manifest fresh (`data/` is gitignored, same reason as SLURP in the old pipeline — see `.gitignore`) via `scripts/prepare_ami_turntaking.py`: downloads real AMI meeting audio, reconstructs real pause durations from the corpus's own `begin_time`/`end_time`/`speaker_id` fields, and labels each gap `MID_TURN` (same speaker resumes) or `TRUE_END` (different speaker starts) — see that script's docstring for the full verification this was based on (why a partial fetch window gives wrong gap durations, why the license is actually cc-by-4.0, etc.).

**On the ≥500 instances / κ≥0.60 gate.** The ≥500-labeled-instances check is real and asserted below. Cohen's κ, as literally specified, does **not** apply here and computing one would be either trivial (1.0, comparing the labeling function to itself) or meaningless (comparing to an arbitrary different heuristic and calling it "reliability"): κ measures agreement between independent human (or otherwise subjective) raters judging the same items. AMI's `TRUE_END`/`MID_TURN` labels are a **deterministic function** of the corpus's own official forced-alignment timestamps and speaker IDs — both already ground truth, not subjective judgments, so there is no second rater to measure agreement against. That gate, as originally specified, describes Phase 1.3's real hand-labeling process (CANDOR + roleplay, still pending) — not this auto-derived stand-in. What's asserted below instead is a **determinism check**: independently recomputing every label from the manifest's own stored timing/speaker fields and confirming it matches the stored label 100% of the time — a real check (it would catch silent corruption or a logic drift bug), just not the same thing as inter-rater κ, and it isn't being called that.

In [ ]:
!python scripts/prepare_ami_turntaking.py

In [ ]:
# --- validation: run the project's own gate script, not a re-implementation of it ---
import subprocess
result = subprocess.run(["python", "scripts/gate_ami_turntaking.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0, "gate_ami_turntaking.py failed -- see output above"
print("[PASS] gate_ami_turntaking.py passed")

In [ ]:
# --- gate: >=500 labeled instances (real, per project spec) + determinism check (NOT Cohen's kappa, see markdown above) ---
import json

with open("data/turn_taking/real_ami/scenarios.jsonl", encoding="utf-8") as f:
    real_scenarios = [json.loads(line) for line in f]

assert len(real_scenarios) >= 500, f"only {len(real_scenarios)} labeled instances, need >=500"
print(f"[PASS] {len(real_scenarios)} labeled instances (>= 500 required)")

# Determinism check: MID_TURN scenarios must have a clip_b (speaker resumes, something follows);
# TRUE_END scenarios must not. This is exactly what the label means by construction -- if it's
# ever violated, the manifest or the labeling logic has drifted out of sync with itself.
mismatches = [
    s["scenario_id"] for s in real_scenarios
    if (s["label"] == "MID_TURN") != (s["clip_b"] is not None)
]
assert not mismatches, f"label/clip_b inconsistency in {len(mismatches)} scenarios: {mismatches[:10]}"
print(f"[PASS] label<->structure determinism check: 0/{len(real_scenarios)} mismatches (this is a consistency check, not Cohen's kappa -- see markdown above)")

In [ ]:
# --- report: label distribution + real pause statistics ---
import numpy as np

n_true_end = sum(s["label"] == "TRUE_END" for s in real_scenarios)
n_mid_turn = len(real_scenarios) - n_true_end
print(f"label distribution: {n_true_end} TRUE_END ({n_true_end/len(real_scenarios)*100:.1f}%), "
      f"{n_mid_turn} MID_TURN ({n_mid_turn/len(real_scenarios)*100:.1f}%)")

meetings = sorted(set(s["meeting_id"] for s in real_scenarios))
print(f"meetings: {meetings}")

for label in ("TRUE_END", "MID_TURN"):
    pauses = np.array([s["pause_ms"] for s in real_scenarios if s["label"] == label])
    print(f"{label} pause_ms: p50={np.percentile(pauses, 50):.0f} p90={np.percentile(pauses, 90):.0f} "
          f"mean={pauses.mean():.0f} min={pauses.min():.0f} max={pauses.max():.0f}")

## 1b. Featurize (real audio, via `src/features/pipeline.py`)

Runs the same Phase 3.2 streaming `FeaturePipeline` used throughout this project (structurally causal, leakage-audited — see `scripts/gate_3_2_leakage_audit.py`) over the real AMI audio. `load_real_scenarios_jsonl` (new, `src/turn_taking/data.py`) adapts the on-disk manifest schema (string labels, `audio_path`, `speech_b_start_sample` — the shape `baseline_fixed_threshold_vad.py`/`ab_compare_endpointers.py` expect) into the in-memory training schema `featurize_scenario` expects (loaded audio array, int label, `pause_end_sample`) — these are two different schemas in this codebase for two different consumers; verified locally on this exact real data before this notebook was written (600/600 scenarios load and featurize with zero NaNs, every scenario has >=1 labeled frame).

In [ ]:
import sys
sys.path.insert(0, "src")

import time
from faster_whisper import WhisperModel
from silero_vad import load_silero_vad
from turn_taking import featurize_scenario, load_real_scenarios_jsonl
from pathlib import Path

vad_model = load_silero_vad(onnx=False)
asr_device = "cuda" if torch.cuda.is_available() else "cpu"
asr_compute_type = "float16" if asr_device == "cuda" else "int8"
asr_model = WhisperModel("small", device=asr_device, compute_type=asr_compute_type)

real_train_val = load_real_scenarios_jsonl(Path("data/turn_taking/real_ami/scenarios.jsonl"), Path("."))
n_train = int(len(real_train_val) * 0.8)
train_scenarios, val_scenarios = real_train_val[:n_train], real_train_val[n_train:]
print(f"{len(train_scenarios)} train scenarios, {len(val_scenarios)} val scenarios")

t0 = time.perf_counter()
train_examples = [featurize_scenario(s, vad_model, asr_model) for s in train_scenarios]
val_examples = [featurize_scenario(s, vad_model, asr_model) for s in val_scenarios]
print(f"featurized {len(train_examples) + len(val_examples)} scenarios in {time.perf_counter() - t0:.1f}s")

torch.save({"train_examples": train_examples, "val_examples": val_examples}, "outputs/ami_featurized.pt")
print("saved -> outputs/ami_featurized.pt (reused by Sections 3 and 4, avoids recomputing)")

In [ ]:
# --- validation ---
# Real data can produce examples with a zero-frame pause window that the synthetic corpus
# never could: sub-100ms real gaps floor-divide to the same frame index on both ends of
# featurize_scenario's [pause_start_frame, pause_end_frame) range. Confirmed this is not a
# rare edge case on this manifest -- 287/600 scenarios (mostly MID_TURN with pause_ms=0.0,
# almost certainly AMI's own transcript segmentation splitting one continuous utterance
# across rows with no real silence between them, not genuine turn-taking holds). Filtered
# here rather than crashing training on them -- a zero-mask example contributes nothing to
# the loss anyway (every loss term in train_model is gated by the mask).
from turn_taking import filter_zero_mask_examples

train_examples, n_skipped_train = filter_zero_mask_examples(train_examples)
val_examples, n_skipped_val = filter_zero_mask_examples(val_examples)
n_skipped = n_skipped_train + n_skipped_val

if n_skipped:
    print(f"WARNING: skipped {n_skipped} scenarios with zero pause-window frames "
          f"({n_skipped_train} train, {n_skipped_val} val) -- see src/turn_taking/data.py's "
          f"filter_zero_mask_examples() docstring for why. The real root cause is "
          f"scripts/prepare_ami_turntaking.py's MIN_GAP_MS=0 allowing zero-duration gaps "
          f"through; raising it would fix this at the source on the next regeneration.")

# Re-save the filtered tensors so a future session reloading outputs/ami_featurized.pt
# (e.g. resuming at Section 3 after a runtime restart) doesn't hit the same crash without
# re-running this cell -- cheap (no re-featurization, just re-serializing what's already
# in memory).
torch.save({"train_examples": train_examples, "val_examples": val_examples}, "outputs/ami_featurized.pt")

all_examples = train_examples + val_examples
for tok, pros, mask, label in all_examples:
    assert tok.shape[0] == pros.shape[0] == mask.shape[0], "sequence length mismatch across tensors"
    assert not torch.isnan(pros).any(), "NaN in prosody features"
    assert mask.sum() > 0, "a scenario has zero labeled (pause-window) frames -- should be unreachable after filtering above"
    assert label in (0, 1)

# NOT lowered to match whatever the real filtered count happens to be -- this is the actual
# floor this project needs to trust the downstream training/ablation/A-B results. If this
# fails, the real fix is regenerating data/turn_taking/real_ami with a higher MIN_GAP_MS in
# scripts/prepare_ami_turntaking.py, not lowering this number.
assert len(all_examples) >= 400, (
    f"only {len(all_examples)} examples remain after filtering {n_skipped} zero-mask scenarios, "
    f"need >=400. See the WARNING above -- consider raising MIN_GAP_MS in "
    f"scripts/prepare_ami_turntaking.py and regenerating instead of lowering this threshold."
)
print(f"[PASS] {len(all_examples)} examples ({len(train_examples)} train / {len(val_examples)} val) "
      f"after filtering {n_skipped} zero-mask, shapes consistent, no NaNs, every scenario has >=1 labeled frame")

## 1c. Save + push Section 1 artifacts

In [ ]:
import shutil
shutil.copy("outputs/ami_featurized.pt", f"{DRIVE_CHECKPOINT_DIR}/ami_featurized.pt")
shutil.copy("data/turn_taking/real_ami/scenarios.jsonl", f"{DRIVE_CHECKPOINT_DIR}/ami_scenarios.jsonl")
print(f"copied featurized data + manifest -> {DRIVE_CHECKPOINT_DIR}")

In [ ]:
# --- validation ---
import os
for fname in ("ami_featurized.pt", "ami_scenarios.jsonl"):
    path = f"{DRIVE_CHECKPOINT_DIR}/{fname}"
    assert os.path.exists(path), f"{path} missing after copy"
print("[PASS] Section 1 artifacts present on Drive")

In [ ]:
# data/turn_taking/real_ami/ is gitignored (regenerable, matches this project's convention for
# every fetched/derived corpus) -- outputs/ami_featurized.pt is the real committed artifact,
# same pattern as outputs/turn_taking_model.pt already being tracked.
!git add outputs/ami_featurized.pt
!git commit -m "Section 1: real AMI turn-taking data, featurized (Colab T4)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per project convention -- do not proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Section 1 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing."
print(f"[PASS] Section 1 pushed successfully, origin/master now at {remote_head[:8]}")

## 2. Fixed-threshold baseline — real AMI audio

The same `scripts/baseline_fixed_threshold_vad.py` used for the Phase 3.1 synthetic-corpus baseline, now pointed at the real AMI manifest via `--labels`, writing to a separate `_real.csv`/`_real.png` (via the `--csv-path`/`--plot-path` overrides added for this run) so the synthetic-corpus baseline isn't overwritten. Verified locally against this exact real data before this notebook was written — real result: interrupt rate plateaus around 20-22% across all four thresholds (200-800ms), a materially different shape than the synthetic corpus's curve (40-80% range), because AMI's real pause distribution skews much shorter than the synthetic corpus's fixed 1500ms TRUE_END trailing silence.

In [ ]:
!python scripts/baseline_fixed_threshold_vad.py --labels data/turn_taking/real_ami/scenarios.jsonl --csv-path outputs/fixed_threshold_vad_baseline_real.csv --plot-path outputs/fixed_threshold_vad_tradeoff_real.png

In [ ]:
# --- validation ---
import pandas as pd

baseline_real_df = pd.read_csv("outputs/fixed_threshold_vad_baseline_real.csv")
assert len(baseline_real_df) == 4, f"expected 4 threshold rows, got {len(baseline_real_df)}"
assert baseline_real_df["response_latency_p50_ms"].notna().all(), "a threshold never fired on any TRUE_END scenario"
assert (baseline_real_df["false_interruption_rate"] >= 0).all() and (baseline_real_df["false_interruption_rate"] <= 1).all()

print("[PASS] real fixed-threshold baseline: 4 thresholds, all fired, rates in [0,1]")
print(baseline_real_df[["threshold_ms", "response_latency_p50_ms", "false_interruption_rate", "n_true_end_missed"]].to_string(index=False))

In [ ]:
shutil.copy("outputs/fixed_threshold_vad_baseline_real.csv", f"{DRIVE_CHECKPOINT_DIR}/fixed_threshold_vad_baseline_real.csv")
shutil.copy("outputs/fixed_threshold_vad_tradeoff_real.png", f"{DRIVE_CHECKPOINT_DIR}/fixed_threshold_vad_tradeoff_real.png")
print(f"copied Section 2 outputs -> {DRIVE_CHECKPOINT_DIR}")

In [ ]:
!git add outputs/fixed_threshold_vad_baseline_real.csv outputs/fixed_threshold_vad_tradeoff_real.png
!git commit -m "Section 2: fixed-threshold baseline on real AMI audio (Colab T4)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per project convention -- do not proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Section 2 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing."
print(f"[PASS] Section 2 pushed successfully, origin/master now at {remote_head[:8]}")

## 3. Train the turn-taking model on real AMI data

Same `TurnTakingGRU` architecture and `train_model` early-stopping loop used for the synthetic-corpus run (added after that first real T4 run overfit badly on ~320 examples -- `patience=5` on val_loss plus `weight_decay` are both still doing real work here on a similarly-sized real training set). `feature_mode="both"` (text + prosody) is the production configuration; Section 4 re-examines that choice on real data instead of assuming the synthetic-corpus ablation finding (prosody-only beat both) transfers.

In [ ]:
from turn_taking import TurnTakingGRU

model = TurnTakingGRU(hidden_dim=128, embed_dim=16, feature_mode="both")
n_params = model.count_params()
print(f"params: {n_params:,}")

In [ ]:
# --- validation ---
assert n_params < 10_000_000, f"{n_params:,} params exceeds the <10M Phase 3.3 target"
print(f"[PASS] {n_params:,} params < 10,000,000")

In [ ]:
from turn_taking import measure_inference_latency_ms

latency_ms = measure_inference_latency_ms(model, device="cuda", n_reps=500)
print(f"single-frame inference latency: {latency_ms:.3f}ms")

In [ ]:
# --- validation ---
assert latency_ms < 5.0, f"{latency_ms:.3f}ms exceeds the <5ms Phase 3.3 target"
print(f"[PASS] {latency_ms:.3f}ms < 5ms")

In [ ]:
from turn_taking import best_epoch, make_batches, train_model

# Reuse the featurized tensors from Section 1 rather than recomputing (ASR is the slow part).
cached = torch.load("outputs/ami_featurized.pt", weights_only=False)
train_examples, val_examples = cached["train_examples"], cached["val_examples"]

train_batches = make_batches(train_examples, batch_size=16)
val_batches = make_batches(val_examples, batch_size=16)

history = train_model(model, train_batches, val_batches, epochs=30, lr=1e-3, patience=5, device="cuda")

In [ ]:
# --- validation ---
import matplotlib.pyplot as plt
from pathlib import Path

best = best_epoch(history)
assert best["val_loss"] < history[0]["val_loss"], (
    "best val_loss never beat epoch 0 -- something is wrong before trusting anything downstream"
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([h["epoch"] for h in history], [h["train_loss"] for h in history], label="train_loss")
ax.plot([h["epoch"] for h in history], [h["val_loss"] for h in history], label="val_loss")
ax.axvline(best["epoch"], color="gray", linestyle="--", alpha=0.6, label=f"best epoch ({best['epoch']}, restored)")
ax.set_xlabel("epoch"); ax.set_ylabel("BCE loss"); ax.legend(); ax.set_title("Real-data (AMI) training curve")
fig.tight_layout()
Path("outputs").mkdir(exist_ok=True)
fig.savefig("outputs/turn_taking_training_curve_real.png", dpi=150)

torch.save(model.state_dict(), "outputs/turn_taking_model_real.pt")

print(f"[PASS] best val_loss {history[0]['val_loss']:.4f} -> {best['val_loss']:.4f} (epoch {best['epoch']}), val_acc={best['val_acc']:.3f}")
print(f"trained {len(history)} epochs before early stopping (cap was 30); restored weights are from epoch {best['epoch']}")
print("saved outputs/turn_taking_model_real.pt and outputs/turn_taking_training_curve_real.png")

In [ ]:
shutil.copy("outputs/turn_taking_model_real.pt", f"{DRIVE_CHECKPOINT_DIR}/turn_taking_model_real.pt")
shutil.copy("outputs/turn_taking_training_curve_real.png", f"{DRIVE_CHECKPOINT_DIR}/turn_taking_training_curve_real.png")
print(f"copied checkpoint + training curve -> {DRIVE_CHECKPOINT_DIR}")

In [ ]:
# --- validation ---
assert os.path.exists(f"{DRIVE_CHECKPOINT_DIR}/turn_taking_model_real.pt"), "checkpoint missing on Drive after copy"
print("[PASS] real checkpoint backed up to Drive")

In [ ]:
!git add outputs/turn_taking_model_real.pt outputs/turn_taking_training_curve_real.png
!git commit -m "Section 3: trained turn-taking model on real AMI data (Colab T4)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per project convention -- do not proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Section 3 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing."
print(f"[PASS] Section 3 pushed successfully, origin/master now at {remote_head[:8]}")

## 4. Ablations on real data

Text-only / prosody-only / both, crossed with a hidden-size sweep — the exact same grid as the synthetic-corpus ablation, on real data this time. The synthetic-corpus finding was prosody-only beating both (text features were noisier than helpful with ~320 training examples); this is re-examined here, not assumed to transfer -- real AMI speech has real ASR partial-token noise of its own, which could go either way.

In [ ]:
import pandas as pd
from turn_taking import TurnTakingGRU as _Model

FEATURE_MODES = ["text", "prosody", "both"]
HIDDEN_SIZES = [32, 128, 512]
ABLATION_EPOCHS = 30

ablation_rows = []
ablation_failures = []

for feature_mode in FEATURE_MODES:
    for hidden_dim in HIDDEN_SIZES:
        m = _Model(hidden_dim=hidden_dim, embed_dim=16, feature_mode=feature_mode)
        n_p = m.count_params()
        h = train_model(m, train_batches, val_batches, epochs=ABLATION_EPOCHS, patience=5, device="cuda", verbose=False)
        best = best_epoch(h)
        lat = measure_inference_latency_ms(m, device="cuda")

        row = {
            "feature_mode": feature_mode,
            "hidden_dim": hidden_dim,
            "params": n_p,
            "best_epoch": best["epoch"],
            "epochs_run": len(h),
            "final_val_loss": best["val_loss"],
            "final_val_acc": best["val_acc"],
            "latency_ms": round(lat, 3),
            "learned_anything": best["val_acc"] > 0.55,
        }
        ablation_rows.append(row)
        if not row["learned_anything"]:
            ablation_failures.append(row)
        print(f"{feature_mode:8s} hidden={hidden_dim:4d}  params={n_p:>8,}  val_acc={row['final_val_acc']:.3f}  val_loss={row['final_val_loss']:.4f}")

ablation_df = pd.DataFrame(ablation_rows)

In [ ]:
# --- validation ---
assert len(ablation_df) == len(FEATURE_MODES) * len(HIDDEN_SIZES), "missing ablation combos"
assert not ablation_df["final_val_loss"].isna().any(), "NaN loss in an ablation arm"
assert (ablation_df["params"] < 10_000_000).all(), "an ablation arm exceeded the 10M param budget"

ablation_df.to_csv("outputs/ablation_results_real.csv", index=False)
print(f"[PASS] {len(ablation_df)} ablation combos completed, saved -> outputs/ablation_results_real.csv")

winner = ablation_df.loc[ablation_df["final_val_acc"].idxmax()]
print(f"\nbest arm on REAL data: {winner['feature_mode']} / hidden={int(winner['hidden_dim'])} (val_acc={winner['final_val_acc']:.3f})")

if ablation_failures:
    print(f"\n{len(ablation_failures)} arm(s) did not learn meaningfully above chance (val_acc <= 0.55):")
    for f in ablation_failures:
        print(f"  {f['feature_mode']} / hidden={f['hidden_dim']}: val_acc={f['final_val_acc']:.3f}")
    print("Record this honestly in the README write-up -- a real negative result on real data is worth more than a hidden one.")

In [ ]:
shutil.copy("outputs/ablation_results_real.csv", f"{DRIVE_CHECKPOINT_DIR}/ablation_results_real.csv")
print(f"copied -> {DRIVE_CHECKPOINT_DIR}")

In [ ]:
!git add outputs/ablation_results_real.csv
!git commit -m "Section 4: ablations on real AMI data (Colab T4)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per project convention -- do not proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Section 4 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing."
print(f"[PASS] Section 4 pushed successfully, origin/master now at {remote_head[:8]}")

## 5. A/B comparison — learned vs. fixed-threshold baseline (headline result)

All 600 real AMI scenarios are used (exceeds the 200-turn minimum). Two steps, not one, and deliberately so:

1. **Sweep first.** `scripts/ab_compare_endpointers.py` sweeps both endpointers across their    full threshold ranges on the real checkpoint + real data, producing a full tradeoff curve.
2. **Discover the matched operating point from that sweep, don't assume one.**    `scripts/eval.py`'s paired bootstrap-CI/Wilcoxon test needs ONE specific    (baseline_threshold, learned_threshold) pair matched by interrupt rate -- its default    values (800ms / 0.7) were tuned for the *synthetic* corpus (`outputs/ab_comparison.csv`)    and reusing them blindly here would reproduce a real bug this project already hit once:    comparing two endpointers at wildly different interrupt-rate operating points and drawing    a conclusion from it (see git history -- caught and fixed during Phase 6 work). The cell    below finds the real matched pair from step 1's sweep instead of assuming the synthetic    corpus's tuned values still apply -- confirmed locally that AMI's baseline curve has a    materially different shape (interrupt rate ~20-22% across ALL thresholds 200-800ms, vs.    40-80% on the synthetic corpus), so this isn't a hypothetical concern.

In [ ]:
!python scripts/ab_compare_endpointers.py --labels data/turn_taking/real_ami/scenarios.jsonl --checkpoint outputs/turn_taking_model_real.pt --csv-path outputs/ab_comparison_real_sweep.csv --plot-path outputs/ab_comparison_real_sweep.png

In [ ]:
# --- validation ---
sweep_df = pd.read_csv("outputs/ab_comparison_real_sweep.csv")
assert (sweep_df["endpointer"] == "fixed_threshold").sum() == 4, "expected 4 baseline threshold rows"
assert (sweep_df["endpointer"] == "learned").sum() == 5, "expected 5 learned threshold rows"
print("[PASS] sweep produced 4 baseline + 5 learned rows")
print(sweep_df[["endpointer", "threshold_label", "response_latency_p50_ms", "false_interruption_rate", "n_true_end_measured"]].to_string(index=False))

In [ ]:
# --- discover the real matched operating point (smallest interrupt-rate gap between the two curves) ---
baseline_rows = sweep_df[sweep_df["endpointer"] == "fixed_threshold"]
learned_rows = sweep_df[sweep_df["endpointer"] == "learned"]

candidates = []
for _, lrow in learned_rows.iterrows():
    # Skip degenerate rows: too few TRUE_END scenarios actually measured to trust, or never fired.
    if pd.isna(lrow["response_latency_p50_ms"]) or lrow["n_true_end_measured"] < 5:
        continue
    gaps = (baseline_rows["false_interruption_rate"] - lrow["false_interruption_rate"]).abs()
    best_idx = gaps.idxmin()
    brow = baseline_rows.loc[best_idx]
    candidates.append((gaps[best_idx], lrow, brow))

assert candidates, "no valid learned-threshold row to match against (all degenerate) -- inspect the sweep above"
candidates.sort(key=lambda c: c[0])
best_gap, chosen_learned, chosen_baseline = candidates[0]

learned_threshold_real = float(chosen_learned["threshold_label"].replace("p>=", ""))
baseline_threshold_ms_real = int(chosen_baseline["threshold_label"].replace("ms", ""))

print(f"matched pair: baseline={baseline_threshold_ms_real}ms (interrupt={chosen_baseline['false_interruption_rate']*100:.1f}%) "
      f"vs learned={learned_threshold_real} (interrupt={chosen_learned['false_interruption_rate']*100:.1f}%), "
      f"gap={best_gap*100:.1f}pp")

In [ ]:
# subprocess.run with an explicit arg list, not `!python ... {var}` shell-magic --
# unambiguous variable passing, doesn't depend on IPython's line-continuation interpolation.
eval_result = subprocess.run(
    [
        "python", "scripts/eval.py",
        "--labels", "data/turn_taking/real_ami/scenarios.jsonl",
        "--checkpoint", "outputs/turn_taking_model_real.pt",
        "--condition", "real_ami",
        "--baseline-threshold-ms", str(baseline_threshold_ms_real),
        "--learned-threshold", str(learned_threshold_real),
        "--csv-path", "outputs/ab_comparison_real.csv",
        "--reference-path", "outputs/ab_comparison_real_reference.json",
        "--update-reference",
    ],
    capture_output=True, text=True,
)
print(eval_result.stdout)
if eval_result.returncode != 0:
    print(eval_result.stderr)
assert eval_result.returncode == 0, "scripts/eval.py failed -- see output above"

In [ ]:
# --- validation + headline result ---
headline_df = pd.read_csv("outputs/ab_comparison_real.csv")
row = headline_df.iloc[-1]

assert row["n_paired_complete_cases"] > 0, "zero paired complete cases -- nothing to report"
print("[PASS] headline A/B result computed on real AMI data\n")

print(f"n paired complete cases: {int(row['n_paired_complete_cases'])} / {int(row['n_true_end'])} TRUE_END scenarios")
print(f"baseline latency p50/p95: {row['baseline_latency_p50_ms']:.0f}ms / {row['baseline_latency_p95_ms']:.0f}ms")
print(f"learned  latency p50/p95: {row['learned_latency_p50_ms']:.0f}ms / {row['learned_latency_p95_ms']:.0f}ms")
print(f"latency delta (learned - baseline): {row['latency_delta_mean_ms']:+.1f}ms, "
      f"95% bootstrap CI [{row['latency_delta_ci95_lo_ms']:+.1f}, {row['latency_delta_ci95_hi_ms']:+.1f}]ms")
print(f"Wilcoxon signed-rank: statistic={row['wilcoxon_statistic']}, p={row['wilcoxon_p_value']}")
print(f"false interruption rate: baseline={row['baseline_false_interruption_rate']*100:.1f}% "
      f"vs learned={row['learned_false_interruption_rate']*100:.1f}%")
print()
if row["latency_delta_ci95_hi_ms"] < 0:
    print("Learned endpointer is faster and the 95% CI excludes zero -- a real, statistically "
          "defensible improvement on this real (AMI) data, at matched interrupt rates.")
elif row["latency_delta_ci95_lo_ms"] > 0:
    print("Learned endpointer is SLOWER here and the CI excludes zero -- a real negative result "
          "on real data. Report it as such; do not paper over it.")
else:
    print("CI includes zero -- no statistically defensible latency difference at this sample size. "
          "Report the CI width honestly rather than picking a side.")

In [ ]:
for fname in ("ab_comparison_real_sweep.csv", "ab_comparison_real_sweep.png", "ab_comparison_real.csv", "ab_comparison_real_reference.json"):
    shutil.copy(f"outputs/{fname}", f"{DRIVE_CHECKPOINT_DIR}/{fname}")
print(f"copied Section 5 outputs -> {DRIVE_CHECKPOINT_DIR}")

In [ ]:
!git add outputs/ab_comparison_real_sweep.csv outputs/ab_comparison_real_sweep.png outputs/ab_comparison_real.csv outputs/ab_comparison_real_reference.json
!git commit -m "Section 5: A/B headline result on real AMI data -- bootstrap CI + Wilcoxon (Colab T4)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per project convention -- do not proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Section 5 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing."
print(f"[PASS] Section 5 pushed successfully, origin/master now at {remote_head[:8]}")

In [ ]:
# Self-contained imports for this section.
import json
import subprocess
from pathlib import Path

import pandas as pd

## 6. Phase 2.1 — real ASR benchmark (full production matrix)

`distil-large-v3`, `large-v3`, `small` × `int8`/`float16` × clean/noisy — the matrix
`scripts/benchmark_asr.py` always supported but that was impractical to run on the CPU laptop
(float16 needs CUDA; large-v3-class models are slow on CPU). Writes to a separate
`outputs/asr_benchmark_gpu.csv` (via `--csv-path`) rather than the existing CPU-smoke
`asr_benchmark.csv`, so the two don't mix in one file.

**Caveat carried forward, not hidden:** the "noisy" condition here still uses SLURP clips +
MUSAN noise (Phase 1.3's existing setup), not CANDOR audio -- CANDOR access is still pending
manual review as of when this section was written. Model-level findings (WER by size/quantization,
RTF, VRAM) are still meaningful now; the noisy-condition WER specifically should be re-measured
once Phase 1.3 is redone against real CANDOR audio.

In [ ]:
!python scripts/prepare_musan_noise.py

In [ ]:
# --- validation ---
noise_dir = Path("data/noise/raw/musan_noise")
n_noise_clips = len(list(noise_dir.glob("*.wav")))
assert n_noise_clips >= 10, f"expected >=10 MUSAN noise clips, found {n_noise_clips}"
print(f"[PASS] {n_noise_clips} MUSAN noise clips on disk")

In [ ]:
!python scripts/snr_sweep.py

In [ ]:
# --- validation ---
manifest_path = Path("data/noise/mixed/manifest.jsonl")
assert manifest_path.exists(), "missing data/noise/mixed/manifest.jsonl -- snr_sweep.py did not run cleanly"
with open(manifest_path) as f:
    n_mixed = sum(1 for _ in f)
assert n_mixed > 0
print(f"[PASS] {n_mixed} SNR-mixed clips ready for the noisy ASR condition")

In [ ]:
!python scripts/benchmark_asr.py --model-sizes distil-large-v3 large-v3 small --compute-types int8 float16 --csv-path outputs/asr_benchmark_gpu.csv

In [ ]:
# --- validation ---
asr_df = pd.read_csv("outputs/asr_benchmark_gpu.csv")

expected_sizes = {"distil-large-v3", "large-v3", "small"}
got_sizes = set(asr_df["model_size"].unique())
assert expected_sizes.issubset(got_sizes), f"missing model sizes: {expected_sizes - got_sizes}"
assert (asr_df["device"] == "cuda").all(), "not every row reports device=cuda"
assert not asr_df["wer"].isna().all(), "no WER values recorded"

print(f"[PASS] {len(asr_df)} real GPU rows in outputs/asr_benchmark_gpu.csv, sizes: {sorted(got_sizes)}")
print()
print(asr_df[["model_size", "compute_type", "condition", "wer", "rtf_p50", "latency_p50_ms", "latency_p95_ms", "vram_used_gb"]].to_string(index=False))

In [ ]:
!git add outputs/asr_benchmark_gpu.csv
!git commit -m "Phase 2.1: real GPU numbers (Colab T4)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per instruction not to proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Phase 2.1 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing to the next section."
print(f"[PASS] Phase 2.1 pushed successfully, origin/master now at {remote_head[:8]}")

## 7. Phase 2.2 — real LLM benchmark (vLLM)

`Qwen2.5-7B-Instruct-AWQ` vs `Qwen2.5-3B-Instruct-AWQ` (bigger-vs-smaller-AWQ, not the
quantized-vs-unquantized comparison this script defaulted to before) × 2 prompt lengths →
TTFT (primary metric; tok/s recorded too but not the focus), plus prefix caching on/off across
the same 6-turn conversation. Neither vLLM nor its model downloads have ever run against a real
install before this -- `scripts/benchmark_llm.py` was written against vLLM's documented
`AsyncLLMEngine` API and syntax-checked only. Report whatever actually happens, including API
mismatches -- that's real information, not a failure to hide.

**Why 4 separate cells instead of 1.** The original single-call form creates up to 4
`vllm.AsyncLLMEngine` instances sequentially in one Python process (7B prompt-lengths, 3B
prompt-lengths, 7B caching-off, 7B caching-on). This is a known vLLM failure mode on
memory-constrained cards -- each engine grabs ~90% of currently-free VRAM for its KV cache,
and that isn't reliably released back when a Python object goes out of scope (see
vllm-project/vllm issues #654 and #14376: "second model requires more memory... No available
memory for the cache blocks"). On a 16GB T4 the single-process form will likely OOM on the
2nd-4th engine. Each cell below runs one engine in its own process instead -- a full process
exit guarantees the GPU memory is actually freed before the next one starts. All 4 append to
the same `outputs/llm_benchmark_gpu.csv` / `outputs/llm_prefix_caching_gpu.csv`, so the
validation cell after them reads the full matrix exactly as before.


In [ ]:
!pip install -q vllm

In [ ]:
# --- validation ---
import vllm
print(f"[PASS] vllm {vllm.__version__} importable")

In [ ]:
!python scripts/benchmark_llm.py --model Qwen/Qwen2.5-7B-Instruct-AWQ --mode prompt-lengths

In [ ]:
# --- validation ---
import pandas as pd
df = pd.read_csv("outputs/llm_benchmark_gpu.csv")
assert "Qwen/Qwen2.5-7B-Instruct-AWQ" in set(df["model"]), "7B prompt-length rows missing"
print("[PASS] 7B prompt-length rows present")

In [ ]:
!python scripts/benchmark_llm.py --model Qwen/Qwen2.5-3B-Instruct-AWQ --mode prompt-lengths

In [ ]:
# --- validation ---
import pandas as pd
df = pd.read_csv("outputs/llm_benchmark_gpu.csv")
assert "Qwen/Qwen2.5-3B-Instruct-AWQ" in set(df["model"]), "3B prompt-length rows missing"
print("[PASS] 3B prompt-length rows present")

In [ ]:
!python scripts/benchmark_llm.py --model Qwen/Qwen2.5-7B-Instruct-AWQ --mode prefix-caching --caching off

In [ ]:
# --- validation ---
import pandas as pd
df = pd.read_csv("outputs/llm_prefix_caching_gpu.csv")
assert (df["enable_prefix_caching"] == False).any(), "caching=off rows missing"
print("[PASS] caching=off rows present")

In [ ]:
!python scripts/benchmark_llm.py --model Qwen/Qwen2.5-7B-Instruct-AWQ --mode prefix-caching --caching on

In [ ]:
# --- validation ---
import pandas as pd
df = pd.read_csv("outputs/llm_prefix_caching_gpu.csv")
assert (df["enable_prefix_caching"] == True).any(), "caching=on rows missing"
print("[PASS] caching=on rows present")

In [ ]:
# --- validation ---
llm_df = pd.read_csv("outputs/llm_benchmark_gpu.csv")
prefix_df = pd.read_csv("outputs/llm_prefix_caching_gpu.csv")

expected_models = {"Qwen/Qwen2.5-7B-Instruct-AWQ", "Qwen/Qwen2.5-3B-Instruct-AWQ"}
got_models = set(llm_df["model"].unique())
assert expected_models == got_models, f"expected {expected_models}, got {got_models}"
assert set(llm_df["prompt_length"].unique()) == {"short", "long"}
assert not llm_df["ttft_p50_ms"].isna().all(), "no TTFT values recorded"
assert set(prefix_df["enable_prefix_caching"].unique()) == {True, False}

print(f"[PASS] {len(llm_df)} prompt-length rows, {len(prefix_df)} prefix-caching rows")
print()
print("TTFT by model x prompt length:")
print(llm_df[["model", "prompt_length", "ttft_p50_ms", "ttft_p95_ms", "tok_s_p50"]].to_string(index=False))
print()
mean_ttft_by_caching = prefix_df[prefix_df["turn_index"] >= 1].groupby("enable_prefix_caching")["ttft_ms"].mean()
print("mean TTFT, turns 1-5 (turn 0 has no shared prefix to reuse), by prefix caching:")
print(mean_ttft_by_caching)

In [ ]:
!git add outputs/llm_benchmark_gpu.csv outputs/llm_prefix_caching_gpu.csv
!git commit -m "Phase 2.2: real GPU numbers (Colab T4)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per instruction not to proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Phase 2.2 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing to the next section."
print(f"[PASS] Phase 2.2 pushed successfully, origin/master now at {remote_head[:8]}")

## 8. Phase 2.3 — real TTS benchmark (Kokoro + Piper, GPU)

Kokoro/Piper already have real, measured CPU numbers from earlier in this project -- this adds
real T4 numbers alongside them. Piper takes an explicit `--use-cuda` flag; Kokoro auto-detects a
GPU via the `onnxruntime-gpu` package (no code change needed, just install it). 3 sentences ×
3 repeats, time-to-first-chunk only.

In [ ]:
!pip uninstall -y onnxruntime -q
!pip install -q onnxruntime-gpu

In [ ]:
# --- validation ---
import onnxruntime
providers = onnxruntime.get_available_providers()
print(f"available onnxruntime providers: {providers}")
assert "CUDAExecutionProvider" in providers, "onnxruntime-gpu install didn't expose CUDAExecutionProvider"
print("[PASS] CUDAExecutionProvider available")

In [ ]:
!python scripts/prepare_tts_models.py

In [ ]:
# --- validation ---
kokoro_model = Path("data/tts/models/kokoro/kokoro-v1.0.int8.onnx")
piper_model = Path("data/tts/models/piper/en_US-lessac-medium.onnx")
assert kokoro_model.exists() and piper_model.exists(), "TTS model files missing"
print("[PASS] Kokoro + Piper model files present")

In [ ]:
!python scripts/benchmark_tts.py --engines kokoro piper --use-cuda

In [ ]:
# --- validation ---
tts_gpu_df = pd.read_csv("outputs/tts_benchmark_gpu.csv")

assert set(tts_gpu_df["engine"].unique()) == {"kokoro", "piper"}, f"expected both engines, got {tts_gpu_df['engine'].unique()}"
assert (tts_gpu_df["device"] == "cuda").all(), "not every row reports device=cuda"

print("[PASS] real GPU TTS benchmark complete, all rows report device=cuda")
print()
print(tts_gpu_df[["engine", "sentence_index", "ttfc_p50_ms", "ttfc_p95_ms"]].to_string(index=False))

try:
    cpu_df = pd.read_csv("outputs/tts_benchmark.csv")
    print("\nmean TTFC p50 by engine, CPU vs GPU (ms):")
    print(pd.DataFrame({"cpu": cpu_df.groupby("engine")["ttfc_p50_ms"].mean(), "gpu": tts_gpu_df.groupby("engine")["ttfc_p50_ms"].mean()}))
except FileNotFoundError:
    print("(no outputs/tts_benchmark.csv from an earlier CPU run in this clone to compare against)")

In [ ]:
!git add outputs/tts_benchmark_gpu.csv
!git commit -m "Phase 2.3: real GPU numbers (Colab T4)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per instruction not to proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Phase 2.3 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing to the next section."
print(f"[PASS] Phase 2.3 pushed successfully, origin/master now at {remote_head[:8]}")

## 9. Phase 0.4 — hardware baseline, T4 side

`scripts/hardware_baseline.py` already has a real CPU row (zenbook-cpu, run locally). This
adds the real Colab T4 row to the same CSV -- VRAM, a synthetic-workload TTFT proxy, and
throughput, so `outputs/hardware_baseline.csv` has both machines this project actually runs
on side by side. No model weights involved, runs in seconds.


In [ ]:
!python scripts/hardware_baseline.py --label colab-t4

In [ ]:
# --- validation ---
import pandas as pd
hb_df = pd.read_csv("outputs/hardware_baseline.csv")
row = hb_df[hb_df["label"] == "colab-t4"]
assert len(row) >= 1, "no colab-t4 row written"
assert (row["device_type"] == "cuda").all(), "colab-t4 row didn't report device_type=cuda"
print("[PASS] colab-t4 row present, device_type=cuda")
print(row.to_string(index=False))

In [ ]:
!git add outputs/hardware_baseline.csv
!git commit -m "Phase 0.4: real T4 hardware baseline (Colab)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per instruction not to proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Phase 0.4 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing."
print(f"[PASS] Phase 0.4 pushed successfully, origin/master now at {remote_head[:8]}")

## Cleanup

Removes the token from git config now that all sections have pushed -- no reason to leave a live credential sitting in `.git/config` for the rest of the session.

In [ ]:
subprocess.run(["git", "remote", "set-url", "origin", "https://github.com/varshitthhh/voice-ai-pipeline.git"], check=True)
print("remote URL reset, token removed from git config")

## Summary

**Real-data turn-taking pipeline (Sections 1-5), on AMI (cc-by-4.0), not synthetic SLURP splices:**
- **Section 1**: real AMI turn-taking manifest (>=500 instances, real pause statistics reported), featurized via the same leakage-audited `FeaturePipeline` used throughout this project.
- **Section 2**: fixed-threshold baseline re-run on real audio -- `outputs/fixed_threshold_vad_baseline_real.csv`.
- **Section 3**: `TurnTakingGRU` trained on real data, early-stopped, checkpoint at `outputs/turn_taking_model_real.pt` (and backed up to Drive).
- **Section 4**: full text/prosody/both x size ablation grid re-run on real data -- `outputs/ablation_results_real.csv` -- any arm that didn't learn meaningfully above chance is flagged, not dropped.
- **Section 5**: the headline result. Real matched-operating-point discovery (not assumed from the synthetic corpus's tuned values), then a real paired bootstrap-CI + Wilcoxon signed-rank test -- `outputs/ab_comparison_real.csv`.

**GPU benchmarks (Sections 6-9), unchanged from the prior session:**
- **Section 6** (Phase 2.1): full production ASR matrix, real WER/RTF/VRAM/p50-p95.
- **Section 7** (Phase 2.2): `Qwen2.5-7B-Instruct-AWQ` vs `Qwen2.5-3B-Instruct-AWQ` on real vLLM, run as 4 separate single-engine processes (known vLLM multi-engine GPU-memory issue on a 16GB T4, vllm-project/vllm #654 / #14376).
- **Section 8** (Phase 2.3): Kokoro + Piper time-to-first-chunk on real T4 GPU.
- **Section 9** (Phase 0.4): real T4 hardware-baseline row alongside the existing real CPU row.

**Still not covered by this notebook:** the real CANDOR/roleplay corpus (Phase 1.1/1.2) -- AMI is a real, license-clean, verified stand-in with real pause timing, not e-commerce content (see Section 0's domain-gap statement). Phase 6.1's three-condition eval set (clean/spontaneous/noisy) also still doesn't exist; Section 5's headline result is one condition, not three.